# Exercises for Chapter 17: Poststratification and missing-data imputation

In [1]:
import os
import sys

import arviz
import bambi
import numpy
import pandas

from scipy import stats

sys.path.append(os.path.abspath("/home/bgawalt/ros"))
import bg_ros

## 17.1, Regression and poststratification

> Section 10.4 presents some models predicting weight from height and other
> variables using survey data in the
> [folder `Earnings`](https://github.com/avehtari/ROS-Examples/tree/master/Earnings/data).
> But these data are not representative of the population. In particular, 62% of
> the respondents in this survey are women, as compared to only 52% of the
> general adult population. We also know the approximate distribution of heights
> in the adult population: normal with mean 63.7 inches and standard deviation 
> 2.7 inches for women, and normal with mean 69.1 inches and standard deviation
> 2.9 inches for men.
>
> (a) Use poststratification to estimate the average weight in the general
>     population, as follows:
> 
> > (i) fit a regression of linear weight on height and sex;
> >
> > (ii) use `posterior_epred` to make predictions for men and women for each
> >     integer value of height from 50 through 80 inches;
> > (iii) poststratify using a  discrete approximation to the normal
> >     distribution for heights given sex, and the known proportion of men and
> >     women in the population.
>
>    Your result should be a set of simulation draws representing the population
>    average weight. Give the median and mad sd of this distribution: this
>    represents your estimate and uncertainty about the population average
>    weight.
>
> (b) Repeat the above steps, this time including the `height:female`
>     interaction in your fitted model before poststratifying.
>
> (c) Repeat (a) and (b), this time performing a regression of log(weight) but
>     still with the goal of estimating average weight in the population, so you
>     will need to exponentiate your predictions in step (ii) before
>     poststratifying.

In [2]:
earn_df = pandas.read_csv('/home/bgawalt/ros/datasets/earnings.csv')[['weight', 'height', 'male']].dropna()
print(bg_ros.dataframe_describe_markdown(earn_df))

|         | weight | height | male
--------- | ------ | ------ | ----
**count** | 1789.00 | 1789.00 | 1789.00
**mean**  | 156.31 | 66.59 | 0.38
**std**   | 34.62 | 3.84 | 0.48
**min**   | 80.00 | 57.00 | 0.00
**25%**   | 130.00 | 64.00 | 0.00
**50%**   | 150.00 | 66.00 | 0.00
**75%**   | 180.00 | 70.00 | 1.00
**max**   | 342.00 | 82.00 | 1.00



### 17.1(a)

In [3]:
# 2-d array; size N x 3;
#   - col 0 is const (intercept)
#   - col 1 is height;
#   - col 2 is gender (1 = male)
census_table = numpy.zeros((2 * (80 - 50 + 1), 3))
prevalence = []
for i, height in enumerate(range(50, 81)):
    # Women's height
    census_table[2 * i, :] = [1, height, 0]
    prevalence.append(
        0.52 * (
            stats.norm.cdf(height + 0.5, loc=63.7, scale=2.7) - 
            stats.norm.cdf(height - 0.5, loc=63.7, scale=2.7)
        )
    )
    census_table[2 * i + 1, :] = [1, height, 1]
    prevalence.append(
        0.48 * (
            stats.norm.cdf(height + 0.5, loc=69.1, scale=2.9) - 
            stats.norm.cdf(height - 0.5, loc=69.1, scale=2.9)
        )
    )
prevalence = numpy.array(prevalence)
print(100 * sum(prevalence))    
print(census_table.shape)
print(census_table.mean(axis=0))

99.9979662790406
(62, 3)
[ 1.  65.   0.5]


In [4]:
weight1_model = bambi.Model('weight ~ height + male', earn_df)
weight1_fit = weight1_model.fit()
print(bg_ros.bambi_markdown(weight1_fit, ['height', 'male']))

Initializing NUTS using jitter+adapt_diag...
/home/bgawalt/miniconda3/envs/ros_conda/lib/python3.14/site-packages/pytensor/link/c/cmodule.py:3004: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, height, male]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 7 seconds.


Coef.     | Mean   | s.e.
--------- | ------ | ------
sigma     | 28.68 | 0.48
Intercept | -106.94 | 16.72
height    | 3.89 | 0.26
male      | 11.83 | 2.01



In [5]:
weight1_coefs = bg_ros.bambi_flatten(weight1_fit, ['Intercept', 'height', 'male'])
print(weight1_coefs.keys())
for arr in weight1_coefs.values():
    assert len(arr) == 4000

dict_keys(['Intercept', 'height', 'male'])


In [6]:
weight1_coefs_arr = numpy.zeros((3, 4000))
for i in range(4000):
    weight1_coefs_arr[0, i] = weight1_coefs['Intercept'][i]
    weight1_coefs_arr[1, i] = weight1_coefs['height'][i]
    weight1_coefs_arr[2, i] = weight1_coefs['male'][i]
epreds1 = census_table.dot(weight1_coefs_arr)
weight_preds1 = prevalence.dot(epreds1)

In [9]:
print(f'Mean: {numpy.mean(weight_preds1):} lbs.\nSE: {numpy.std(weight_preds1)} lbs.')

Mean: 156.34073292611527 lbs.
SE: 0.7247453807139882 lbs.


### 17.1(b)

In [11]:
census_table_inter = numpy.zeros((62, 4))
census_table_inter[:, 0:3] = census_table
census_table_inter[:, 3] = census_table[:, 1] * census_table[:, 2]
print(census_table_inter.mean(axis=0))

[ 1.  65.   0.5 32.5]


In [12]:
weight2_model = bambi.Model('weight ~ height + male + height:male', earn_df)
weight2_fit = weight2_model.fit()
print(bg_ros.bambi_markdown(weight2_fit, ['height', 'male', 'height:male']))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, height, male, height:male]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 16 seconds.


Coef.       | Mean   | s.e.
----------- | ------ | ------
sigma       | 28.60 | 0.48
Intercept   | -61.39 | 21.15
height      | 3.18 | 0.33
male        | -96.80 | 32.56
height:male | 1.61 | 0.48



In [16]:
weight2_coefs = bg_ros.bambi_flatten(weight1_fit, ['Intercept', 'height', 'male', 'height:male'])
print(weight2_coefs.keys())
for arr in weight1_coefs.values():
    assert len(arr) == 4000
weight2_coefs_arr = numpy.zeros((4, 4000))
for i in range(4000):
    weight2_coefs_arr[0, i] = weight2_coefs['Intercept'][i]
    weight2_coefs_arr[1, i] = weight2_coefs['height'][i]
    weight2_coefs_arr[2, i] = weight2_coefs['male'][i]
    weight2_coefs_arr[3, i] = weight2_coefs['height:male'][i]
epreds2 = census_table_inter.dot(weight2_coefs_arr)
weight_preds2 = prevalence.dot(epreds2)

dict_keys(['Intercept', 'height', 'male', 'height:male'])


In [17]:
print(f'Mean: {numpy.mean(weight_preds2):} lbs.\nSE: {numpy.std(weight_preds2)} lbs.')

Mean: 156.2278181688758 lbs.
SE: 0.7149364912067607 lbs.


### 17.1(c)

In [22]:
weight3_model = bambi.Model('log(weight) ~ height + male', earn_df)
weight3_fit = weight3_model.fit()
print(bg_ros.bambi_markdown(weight3_fit, ['height', 'male']))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, height, male]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 7 seconds.


Coef.     | Mean   | s.e.
--------- | ------ | ------
sigma     | 0.17 | 0.00
Intercept | 3.37 | 0.10
height    | 0.03 | 0.00
male      | 0.08 | 0.01



In [23]:
weight3_coefs = bg_ros.bambi_flatten(weight3_fit, ['Intercept', 'height', 'male'])
print(weight3_coefs.keys())
for arr in weight3_coefs.values():
    assert len(arr) == 4000
weight3_coefs_arr = numpy.zeros((3, 4000))
for i in range(4000):
    weight3_coefs_arr[0, i] = weight3_coefs['Intercept'][i]
    weight3_coefs_arr[1, i] = weight3_coefs['height'][i]
    weight3_coefs_arr[2, i] = weight3_coefs['male'][i]
epreds3 = numpy.exp(census_table.dot(weight3_coefs_arr))
weight_preds3 = prevalence.dot(epreds3)
print(f'Mean: {numpy.mean(weight_preds3):} lbs.\nSE: {numpy.std(weight_preds3)} lbs.')

dict_keys(['Intercept', 'height', 'male'])
Mean: 154.02948371956447 lbs.
SE: 0.6954753308168531 lbs.


In [24]:
weight4_model = bambi.Model('log(weight) ~ height + male + height:male', earn_df)
weight4_fit = weight4_model.fit()
print(bg_ros.bambi_markdown(weight4_fit, ['height', 'male', 'height:male']))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, height, male, height:male]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 16 seconds.


Coef.       | Mean   | s.e.
----------- | ------ | ------
sigma       | 0.17 | 0.00
Intercept   | 3.54 | 0.13
height      | 0.02 | 0.00
male        | -0.32 | 0.19
height:male | 0.01 | 0.00



In [25]:
weight4_coefs = bg_ros.bambi_flatten(weight4_fit, ['Intercept', 'height', 'male', 'height:male'])
print(weight4_coefs.keys())
for arr in weight4_coefs.values():
    assert len(arr) == 4000
weight4_coefs_arr = numpy.zeros((4, 4000))
for i in range(4000):
    weight4_coefs_arr[0, i] = weight4_coefs['Intercept'][i]
    weight4_coefs_arr[1, i] = weight4_coefs['height'][i]
    weight4_coefs_arr[2, i] = weight4_coefs['male'][i]
    weight4_coefs_arr[3, i] = weight4_coefs['height:male'][i]
epreds4 = numpy.exp(census_table_inter.dot(weight4_coefs_arr))
weight_preds4 = prevalence.dot(epreds4)
print(f'Mean: {numpy.mean(weight_preds4):} lbs.\nSE: {numpy.std(weight_preds4)} lbs.')

dict_keys(['Intercept', 'height', 'male', 'height:male'])
Mean: 153.94637980009884 lbs.
SE: 0.6566203545867645 lbs.
